# 11 — チューニングと数式変更の実践手順

## 最終到達目標
現象を「どの層の、どの式の、どの残差か」へ戻し、一度に1仮説だけ変更する。
この章は魔法の推奨値ではなく、再現可能な変更手順を作る。

照合対象: [`legged_control` `a7f381c036`](https://github.com/qiayuanliao/legged_control/tree/a7f381c0367e98e31c01336e678eef47e304d40d)


In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists():
        ROOT = candidate
        break

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True})
print("repository:", ROOT)


repository: /home/takuya/work/mpc_dog


## 変更の順序
1. baseline commit/config、robot、gait、指令、床、seedを固定
2. 失敗を層へ分類: sensor/frame → estimator → reference/gait → NMPC → WBC → joint/HW
3. 数式の単位・符号・shapeを手計算と静的testで確認
4. 変更parameterは1群、式変更は1項だけ
5. constraint residual、solver status/time、tracking、torque、slipを保存
6. 改善と副作用を比較し、戻せる差分にする

### 症状から最初に見る場所
- 静止でbase位置drift: IMU重力/frame、接触flag、KF noise
- 横滑り: 実摩擦、NMPC円錐margin、WBC pyramid、接触誤判定
- 足先が遅れる: swing task residual、WBC weight、torque saturation
- 姿勢追従が弱い: reference、Q、feasibility、base task weight
- torque振動: policy age、接触切替、WBC active set、Kd、delay


In [2]:
# 変更記録を機械的に比較するための最小schema。
baseline = {
    "mu": 0.3, "horizon_s": 1.0, "mpc_hz": 100,
    "wbc_weight_swing": 100.0,
    "wbc_weight_base": 1.0,
    "wbc_weight_force": 0.01,
    "joint_kp": 0.0, "joint_kd": 3.0,
}
trial = baseline | {"wbc_weight_force": 0.03}
changed = {k: (baseline[k], trial[k]) for k in baseline if baseline[k] != trial[k]}
assert len(changed) == 1, "一度に1仮説の規則に反している"
changed


{'wbc_weight_force': (0.01, 0.03)}

## 数式変更テンプレート

例: 摩擦を等方円錐から異方性ellipseへ変えるなら
\[
\sqrt{(F_x/\mu_x)^2+(F_y/\mu_y)^2}\le F_z
\]
と書き、次を同時に定義する。

- \(\mu_x,\mu_y\) の物理的意味と同定法
- \(F_z<0\) を許さない条件
- smooth化epsilonとgradient
- NMPC側soft constraintとWBC側linear approximationの整合
- flat floorで元式へ戻る回帰test

「式を変える」はC++1行の変更ではなく、model・constraint・solver微分・下位実行・
testの契約変更である。


In [3]:
# 等方円錐と異方性ellipseのmarginを比較する。
def isotropic_margin(fx, fy, fz, mu):
    return mu*fz - np.hypot(fx, fy)

def anisotropic_margin(fx, fy, fz, mux, muy):
    return fz - np.sqrt((fx/mux)**2 + (fy/muy)**2)

test_forces = np.array([[10, 0, 50], [0, 10, 50], [12, 8, 50], [20, 0, 50]])
for f in test_forces:
    old = isotropic_margin(*f, mu=0.3)
    new = anisotropic_margin(*f, mux=0.4, muy=0.2)
    print(f"F={f}: isotropic={old:7.3f}, anisotropic={new:7.3f}")


F=[10  0 50]: isotropic=  5.000, anisotropic= 25.000
F=[ 0 10 50]: isotropic=  5.000, anisotropic=  0.000
F=[12  8 50]: isotropic=  0.578, anisotropic=  0.000
F=[20  0 50]: isotropic= -5.000, anisotropic=  0.000


## 最終演習（順番を守る）
1. `task.info`のQ/Rを全24状態・24入力へ対応付ける。
2. 静止4脚でweight-compensating inputと力学残差を計算する。
3. trot 2脚支持でNMPC円錐とWBC pyramid双方のmarginを出す。
4. WBCの各task residualをlogできる設計を作る。
5. policy age watchdogとQP failure fallbackを設計する。
6. Qを1群だけ変え、追従・constraint・torque・solve timeを比較する。
7. 最後に、摩擦または遊脚軌道の式を1つ変更し、元式へ戻る回帰testを書く。

## 修了判定
次を説明できれば、コード変更へ進める。

- x/u/rbd/WBC変数の中身、単位、frame
- Gait、NMPC、WBC、hybrid jointの責務境界
- KFが推定するもの/しないもの
- NMPC円錐とWBC pyramidの差
- weighted WBCでGRF目標がずれる理由
- 100/500 Hz間でpolicyが古くなる危険
- parameter変更と数式変更の検証項目


## 章固有の背景
                複数層のparameterを同時変更すると、改善原因も副作用も同定できない。

                ## 章固有の目的
                症状を残差へ戻し、config変更と式変更を回帰可能な実験として設計する。

                ## この章のASCIIデータフロー
                ```text
                symptom -> identify block/residual -> freeze baseline -> one change
 -> unit/shape/gradient tests -> 30-scenario evidence -> accept or revert
                ```

                ## 上流C++ / faithful pseudocode と数式の行対応
                ```cpp
                // external/legged_control/legged_controllers/config/a1/task.info
Q(state)=...; R(input)=...; mu=0.3; // parameter変更: cost/feasible set
// LeggedInterface::setupOptimalControlProblem
constraint = FrictionConeConstraint(...);     // NMPC式を変更する場所
// WbcBase::formulateFrictionConeTask
D_i * f_i <= 0;                               // WBC近似も同時に整合
// 検査式
old_margin=mu*Fz-hypot(Fx,Fy);                // baseline残差
new_margin=Fz-sqrt((Fx/mux)^2+(Fy/muy)^2);    // 新式と単位を明示
                ```

                **事実のラベル**: `external/legged_control/` の記述はcommit
                `a7f381c0367e98e31c01336e678eef47e304d40d` の上流実装事実。数式展開はそのinterfaceを説明する理論。
                `src/legged_control_mujoco/` に言及した行はproject所有adapterの実装であり、
                ROS1/OCS2 SQP原実装とは同一ではない。

                ## 章固有の結論
                baseline固定、1仮説、物理残差、solver/time、安全指標を揃えて初めて調整になる。式変更はNMPCとWBC双方の整合まで含む。
